# Data Pipeline Exploration

Quick look at the datasets and how the low-label subsampling works.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from src.data import load_dataset_by_name, subsample_low_label
from collections import Counter

## 1. Load Banking77

In [2]:
banking = load_dataset_by_name("banking77")
print(f"Dataset:     {banking.name}")
print(f"Classes:     {banking.num_classes}")
print(f"Train size:  {len(banking.train)}")
print(f"Test size:   {len(banking.test)}")
print(f"Columns:     text='{banking.text_column}', label='{banking.label_column}'")
print(f"\nFirst 10 class names:")
for i, name in enumerate(banking.label_names[:10]):
    print(f"  {i}: {name}")

Using the latest cached version of the dataset since PolyAI/banking77 couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at /Users/omertugrul/.cache/huggingface/datasets/PolyAI___banking77/default/1.1.0/17ffc2ed47c2ed928bee64127ff1dbc97204cb974c2f980becae7c864007aed9 (last modified on Sat Feb 21 22:52:59 2026).


Dataset:     banking77
Classes:     77
Train size:  10003
Test size:   3080
Columns:     text='text', label='label'

First 10 class names:
  0: activate_my_card
  1: age_limit
  2: apple_pay_or_google_pay
  3: atm_support
  4: automatic_top_up
  5: balance_not_updated_after_bank_transfer
  6: balance_not_updated_after_cheque_or_cash_deposit
  7: beneficiary_not_allowed
  8: cancel_transfer
  9: card_about_to_expire


## 2. Look at some examples

In [3]:
for i in range(5):
    ex = banking.train[i]
    label_id = ex[banking.label_column]
    label_name = banking.label_names[label_id]
    print(f"[{label_id:2d}] {label_name}")
    print(f"     \"{ex[banking.text_column]}\"")
    print()

[11] card_arrival
     "I am still waiting on my card?"

[11] card_arrival
     "What can I do if my card still hasn't arrived after 2 weeks?"

[11] card_arrival
     "I have been waiting over a week. Is the card still coming?"

[11] card_arrival
     "Can I track my card while it is in the process of delivery?"

[11] card_arrival
     "How do I know if I will get my card, or if it is lost?"



## 3. Class distribution in the full training set

In [4]:
dist = Counter(banking.train[banking.label_column])
counts = sorted(dist.values())
print(f"Min examples/class: {counts[0]}")
print(f"Max examples/class: {counts[-1]}")
print(f"Mean:               {sum(counts)/len(counts):.1f}")
print(f"Total classes:      {len(dist)}")

Min examples/class: 35
Max examples/class: 187
Mean:               129.9
Total classes:      77


## 4. Low-label subsampling (L=10, seed=42)

In [5]:
splits = subsample_low_label(banking, labels_per_class=10, seed=42)

print(f"labeled_train: {len(splits.labeled_train)} examples")
print(f"labeled_cal:   {len(splits.labeled_cal)} examples")
print(f"unlabeled:     {len(splits.unlabeled)} examples")
print(f"total:         {len(splits.labeled_train) + len(splits.labeled_cal) + len(splits.unlabeled)}")
print(f"\nExpected labeled: 10 × 77 = {10 * 77}")
print(f"Actual labeled:   {len(splits.labeled_train) + len(splits.labeled_cal)}")

labeled_train: 539 examples
labeled_cal:   231 examples
unlabeled:     9233 examples
total:         10003

Expected labeled: 10 × 77 = 770
Actual labeled:   770


## 5. Per-class breakdown (train vs calibration)

In [6]:
train_dist = Counter(splits.labeled_train[banking.label_column])
cal_dist = Counter(splits.labeled_cal[banking.label_column])

print(f"{'Class':<6} {'Name':<30} {'Train':>5} {'Cal':>5} {'Total':>5}")
print("-" * 55)
for cls in range(min(15, banking.num_classes)):
    n_tr = train_dist.get(cls, 0)
    n_cal = cal_dist.get(cls, 0)
    print(f"{cls:<6} {banking.label_names[cls]:<30} {n_tr:>5} {n_cal:>5} {n_tr+n_cal:>5}")
print(f"{'...':<6}")
print(f"{'All':<37} {sum(train_dist.values()):>5} {sum(cal_dist.values()):>5} {sum(train_dist.values())+sum(cal_dist.values()):>5}")

Class  Name                           Train   Cal Total
-------------------------------------------------------
0      activate_my_card                   7     3    10
1      age_limit                          7     3    10
2      apple_pay_or_google_pay            7     3    10
3      atm_support                        7     3    10
4      automatic_top_up                   7     3    10
5      balance_not_updated_after_bank_transfer     7     3    10
6      balance_not_updated_after_cheque_or_cash_deposit     7     3    10
7      beneficiary_not_allowed            7     3    10
8      cancel_transfer                    7     3    10
9      card_about_to_expire               7     3    10
10     card_acceptance                    7     3    10
11     card_arrival                       7     3    10
12     card_delivery_estimate             7     3    10
13     card_linking                       7     3    10
14     card_not_working                   7     3    10
...   
All           

## 6. Examples from the labeled train split

In [7]:
print("=== Labeled TRAIN examples (used for student CE loss) ===\n")
for i in range(5):
    ex = splits.labeled_train[i]
    label_id = ex[banking.label_column]
    print(f"  [{banking.label_names[label_id]}]")
    print(f"  \"{ex[banking.text_column]}\"\n")

=== Labeled TRAIN examples (used for student CE loss) ===

  [activate_my_card]
  "i cant seem to activate card"

  [activate_my_card]
  "how long should i wait ti activate my card"

  [activate_my_card]
  "My new card has arrived, what's the activation procedure?"

  [activate_my_card]
  "I just got a new card how do I get it to start working?"

  [activate_my_card]
  "Show me how to activate my card"



In [8]:
print("=== Labeled CALIBRATION examples (used for router training) ===\n")
for i in range(5):
    ex = splits.labeled_cal[i]
    label_id = ex[banking.label_column]
    print(f"  [{banking.label_names[label_id]}]")
    print(f"  \"{ex[banking.text_column]}\"\n")

=== Labeled CALIBRATION examples (used for router training) ===

  [activate_my_card]
  "Can I have my card activated?"

  [activate_my_card]
  "How can I activate the new card i got?"

  [activate_my_card]
  "Could you please tell me how to activate my new card?"

  [age_limit]
  "I'd like to open an account for my kids can I do that?"

  [age_limit]
  "Do you have any accounts for children ?"



## 7. Reproducibility check

In [9]:
s1 = subsample_low_label(banking, labels_per_class=10, seed=42)
s2 = subsample_low_label(banking, labels_per_class=10, seed=42)
s3 = subsample_low_label(banking, labels_per_class=10, seed=99)

same_seed = s1.labeled_train[banking.text_column] == s2.labeled_train[banking.text_column]
diff_seed = s1.labeled_train[banking.text_column] != s3.labeled_train[banking.text_column]

print(f"Same seed (42 vs 42) gives identical splits: {same_seed}")
print(f"Diff seed (42 vs 99) gives different splits: {diff_seed}")

Same seed (42 vs 42) gives identical splits: True
Diff seed (42 vs 99) gives different splits: True


## 8. Quick look at AG News

In [10]:
ag = load_dataset_by_name("ag_news")
print(f"Dataset:     {ag.name}")
print(f"Classes:     {ag.num_classes}  {ag.label_names}")
print(f"Train size:  {len(ag.train)}")
print(f"Test size:   {len(ag.test)}")

print("\nSample examples:")
for i in range(3):
    ex = ag.train[i]
    label_name = ag.label_names[ex[ag.label_column]]
    text = ex[ag.text_column][:120] + "..." if len(ex[ag.text_column]) > 120 else ex[ag.text_column]
    print(f"  [{label_name}] {text}\n")

Dataset:     ag_news
Classes:     4  ['World', 'Sports', 'Business', 'Sci/Tech']
Train size:  120000
Test size:   7600

Sample examples:
  [Business] Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics,...

  [Business] Carlyle Looks Toward Commercial Aerospace (Reuters) Reuters - Private investment firm Carlyle Group,\which has a reputat...

  [Business] Oil and Economy Cloud Stocks' Outlook (Reuters) Reuters - Soaring crude prices plus worries\about the economy and the ou...



In [11]:
ag_splits = subsample_low_label(ag, labels_per_class=10, seed=42)

print(f"AG News with L=10:")
print(f"  labeled_train: {len(ag_splits.labeled_train)} (expected: 4 × 7 = 28)")
print(f"  labeled_cal:   {len(ag_splits.labeled_cal)} (expected: 4 × 3 = 12)")
print(f"  unlabeled:     {len(ag_splits.unlabeled)}")
print(f"  total:         {len(ag_splits.labeled_train) + len(ag_splits.labeled_cal) + len(ag_splits.unlabeled)}")

AG News with L=10:
  labeled_train: 28 (expected: 4 × 7 = 28)
  labeled_cal:   12 (expected: 4 × 3 = 12)
  unlabeled:     119960
  total:         120000


## 9. Compare L=10 vs L=30 for Banking77

In [12]:
for L in [10, 20, 30]:
    s = subsample_low_label(banking, labels_per_class=L, seed=42)
    pct_labeled = (len(s.labeled_train) + len(s.labeled_cal)) / len(banking.train) * 100
    print(f"L={L:2d}:  train={len(s.labeled_train):5d}  cal={len(s.labeled_cal):4d}  "
          f"unlabeled={len(s.unlabeled):5d}  ({pct_labeled:.1f}% labeled)")

L=10:  train=  539  cal= 231  unlabeled= 9233  (7.7% labeled)
L=20:  train= 1078  cal= 462  unlabeled= 8463  (15.4% labeled)
L=30:  train= 1617  cal= 693  unlabeled= 7693  (23.1% labeled)
